In [1]:
import openassetpricing as oap
from tqdm import tqdm
import pandas as pd
import numpy as np

In [2]:
openap = oap.OpenAP()

In [3]:
signal_doc = openap.dl_signal_doc('pandas')

In [4]:
all_features = signal_doc.Acronym.tolist()

In [5]:
import wrds

db = wrds.Connection(wrds_username='francoisgoybet')

query = f"""
select
    permno,
    date,
    ret,
    prc,
    shrout,
    abs(prc*shrout)/1e6 as market_cap_musd
from crsp.msf
where abs(prc*shrout)/1e6 > 100
and date >= '1990-01-01'
order by permno, date
"""

crsp_df = db.raw_sql(query)
crsp_df['date'] = pd.to_datetime(crsp_df['date'])

crsp_df['yyyymm'] = crsp_df['date'].dt.strftime('%Y%m')
crsp_df = crsp_df[['permno', 'yyyymm', 'ret']]

crsp_df['yyyymm'] = crsp_df['yyyymm'].astype(str)  # ensure string for merge
crsp_df = crsp_df.sort_values(['permno', 'yyyymm'])

crsp_df['ret_1m'] = crsp_df.groupby('permno')['ret'].shift(-1)
crsp_df['ret_3m'] = crsp_df.groupby('permno')['ret'].shift(-3)
crsp_df['ret_6m'] = crsp_df.groupby('permno')['ret'].shift(-6)

Loading library list...
Done


In [6]:
import pandas as pd
from tqdm import tqdm
import openassetpricing as oap

# -----------------------------
# INIT
# -----------------------------
openap = oap.OpenAP()

base = crsp_df.copy()
base['permno'] = base['permno'].astype(int)
base['yyyymm'] = base['yyyymm'].astype(int)

signals = signal_doc.Acronym.tolist()

chunk_size = 20

output_prefix = "ml_dataset_part_"

# -----------------------------
# FUNCTION
# -----------------------------
def load_features(batch_signals):
    df = openap.dl_signal('pandas', batch_signals)

    df['permno'] = df['permno'].astype(int)
    df['yyyymm'] = df['yyyymm'].astype(int)

    return df

# -----------------------------
# MAIN LOOP (MULTI FILE OUTPUT)
# -----------------------------
for i in tqdm(range(0, len(signals), chunk_size), desc="OSAP batching"):
    if i < 120:
        continue
    print(f"Processing signals {i} to {min(i+chunk_size, len(signals))}...")
    batch = signals[i:i+chunk_size]

    feat = load_features(batch)

    # SAFE LEFT MERGE
    base_chunk = base.merge(
        feat,
        on=['permno', 'yyyymm'],
        how='left'
    )

    # -----------------------------
    # 💾 SAVE EACH CHUNK SEPARATELY
    # -----------------------------
    base_chunk.to_parquet(
        f"{output_prefix}{i:04d}.parquet",
        index=False
    )

    del feat, base_chunk

print("DONE — all chunks saved")

OSAP batching:   0%|          | 0/17 [00:00<?, ?it/s]

Processing signals 120 to 140...
WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\franc\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
One or more input predictors are not available.

Data is downloaded: 2 mins


OSAP batching:  41%|████      | 7/17 [01:59<02:50, 17.03s/it]

Processing signals 140 to 160...
WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\franc\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


OSAP batching:  47%|████▋     | 8/17 [09:37<13:45, 91.75s/it]

...

One or more input predictors are not available.

Data is downloaded: 8s
Processing signals 180 to 200...
One or more input predictors are not available.

Data is downloaded: 18s


OSAP batching:  59%|█████▉    | 10/17 [10:03<07:20, 62.92s/it]

Processing signals 200 to 220...


OSAP batching: 100%|██████████| 17/17 [10:10<00:00, 35.89s/it]

One or more input predictors are not available.

Data is downloaded: 7s
Processing signals 220 to 240...
One or more input predictors are not available.

Data is downloaded: 0s
Processing signals 240 to 260...
One or more input predictors are not available.

Data is downloaded: 0s
Processing signals 260 to 280...
One or more input predictors are not available.

Data is downloaded: 0s
Processing signals 280 to 300...
One or more input predictors are not available.

Data is downloaded: 0s
Processing signals 300 to 320...
One or more input predictors are not available.

Data is downloaded: 0s
Processing signals 320 to 331...
One or more input predictors are not available.

Data is downloaded: 0s
DONE — all chunks saved


In [9]:
df = pd.read_parquet("ml_dataset_part_0020.parquet")

In [7]:
import pandas as pd
import glob

files = glob.glob("ml_dataset_part_*.parquet")

dfs = [pd.read_parquet(f) for f in files]
# merge them on permno	yyyymm	ret	ret_1m	ret_3m	ret_6m
# we can use reduce to merge them iteratively
from functools import reduce
from functools import reduce
def merge_dfs(left, right):
    return pd.merge(left, right, on=['permno', 'yyyymm', 'ret', 'ret_1m', 'ret_3m', 'ret_6m'], how='outer')
final_df = reduce(merge_dfs, dfs).drop_duplicates(subset=['permno', 'yyyymm']) 

In [8]:
# check the percenage of missing values in each column
missing_percentage = final_df.isnull().mean() * 100
print(missing_percentage.sort_values(ascending=False))
# check number and proportiion of columns with more than x% missing values
threshold = 20
num_cols_above_threshold = (missing_percentage > threshold).sum()
total_cols = len(missing_percentage)
proportion_above_threshold = num_cols_above_threshold / total_cols * 100
print(f"Number of columns with more than {threshold}% missing values: {num_cols_above_threshold}")
print(f"Proportion of columns with more than {threshold}% missing values: {proportion_above_threshold:.2f}%")

ChNAnalyst     100.000000
CitationsRD    100.000000
IndRetBig      100.000000
EarnSupBig     100.000000
Activism1       99.874390
                  ...    
betaVIX          0.014778
IdioVol3F        0.014778
MaxRet           0.007389
permno           0.000000
yyyymm           0.000000
Length: 110, dtype: float64
Number of columns with more than 20% missing values: 32
Proportion of columns with more than 20% missing values: 29.09%


In [15]:
df.to_parquet("final_ml_dataset.parquet", index=False)

In [16]:
signal_doc.to_parquet("../data/signal_doc.parquet", index=False)
